In [40]:
import pandas as pd
import requests, io

In [41]:
api_url="https://data.moenv.gov.tw/api/v2/aqx_p_02?api_key=4c89a32a-a214-461b-bf29-30ff32a61a8a&limit=1000&sort=datacreationdate%20desc&format=CSV"

In [42]:
resp = requests.get(api_url, verify=False)   #SSL 憑證驗證失敗,無法直接使用 pd.read_csv(api_url)，所以需使用正規方式呼叫檔案
df = pd.read_csv(io.StringIO(resp.text))     #使用 io.StringIO()將url包裝成一個檔案，騙過read_csv
df

c:\Users\shan4\pm25\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'data.moenv.gov.tw'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


,site,county,pm25,datacreationdate,itemunit
0,林森,臺南市,14.0,2026-05-09 17:00,μg/m3
1,員林,彰化縣,14.0,2026-05-09 17:00,μg/m3
2,臺灣大道,臺中市,9.0,2026-05-09 17:00,μg/m3
3,大城,彰化縣,9.0,2026-05-09 17:00,μg/m3
4,富貴角,新北市,3.0,2026-05-09 17:00,μg/m3
...,...,...,...,...,...
995,臺南,臺南市,16.0,2026-05-09 05:00,μg/m3
996,安南,臺南市,17.0,2026-05-09 05:00,μg/m3
997,善化,臺南市,12.0,2026-05-09 05:00,μg/m3
998,新營,臺南市,22.0,2026-05-09 05:00,μg/m3


In [43]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   site              1000 non-null   str    
 1   county            1000 non-null   str    
 2   pm25              986 non-null    float64
 3   datacreationdate  1000 non-null   str    
 4   itemunit          1000 non-null   str    
dtypes: float64(1), str(4)
memory usage: 39.2 KB


In [44]:
df.describe()

,pm25
count,986.000000
mean,10.401623
std,7.210888
min,0.000000
25%,5.000000
50%,9.000000
75%,15.000000
max,60.000000


### 清理資料跟刪除重複

In [45]:
df.head(5)   #選取前5筆資料

,site,county,pm25,datacreationdate,itemunit
0,林森,臺南市,14.0,2026-05-09 17:00,μg/m3
1,員林,彰化縣,14.0,2026-05-09 17:00,μg/m3
2,臺灣大道,臺中市,9.0,2026-05-09 17:00,μg/m3
3,大城,彰化縣,9.0,2026-05-09 17:00,μg/m3
4,富貴角,新北市,3.0,2026-05-09 17:00,μg/m3


### 確認重複資料

In [47]:
df2=df[df.duplicated(subset=["site","datacreationdate"])]    #取值用中括號[],函數用小括號()
df2

,site,county,pm25,datacreationdate,itemunit


### subset 約定重複跟nan的依據

In [48]:
#刪除重複
df1=df.drop_duplicates(subset=["site","datacreationdate"]).dropna()    #.dropna() 刪除空值
df

,site,county,pm25,datacreationdate,itemunit
0,林森,臺南市,14.0,2026-05-09 17:00,μg/m3
1,員林,彰化縣,14.0,2026-05-09 17:00,μg/m3
2,臺灣大道,臺中市,9.0,2026-05-09 17:00,μg/m3
3,大城,彰化縣,9.0,2026-05-09 17:00,μg/m3
4,富貴角,新北市,3.0,2026-05-09 17:00,μg/m3
...,...,...,...,...,...
995,臺南,臺南市,16.0,2026-05-09 05:00,μg/m3
996,安南,臺南市,17.0,2026-05-09 05:00,μg/m3
997,善化,臺南市,12.0,2026-05-09 05:00,μg/m3
998,新營,臺南市,22.0,2026-05-09 05:00,μg/m3


### sqlite 建立資料庫

In [50]:
import sqlite3

In [55]:
# unique 插入資料唯一的約束
sqlstr='''
create table if not exists data(
id integer primary key autoincrement,
site text,
county text,
pm25 integer,
datacreationdate text,
itemunit text,
unique(site,datacreationdate)
)
'''

In [58]:
conn=sqlite3.connect("pm25.db")
cursor=conn.cursor()

conn,cursor

(<sqlite3.Connection at 0x286b5e78e50>, <sqlite3.Cursor at 0x286b5a377c0>)

In [59]:
cursor.execute(sqlstr)
conn.commit()

### 插入資料